# turboquant-gpu

5.02x KV cache compression for LLM inference using cuTile kernels with PyTorch fallback.

Paper: [TurboQuant](https://arxiv.org/abs/2501.09747) (ICLR 2026)

## install

Detects your CUDA driver version and installs a compatible PyTorch wheel. Also installs `cuda-tile` for GPU kernel acceleration (falls back to PyTorch if unavailable). Restart the kernel after this cell finishes.

In [ ]:
import subprocess, sys, ctypes

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", *args], check=True)

subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"],
               capture_output=True)

try:
    libcuda = ctypes.CDLL("libcuda.so.1")
    v = ctypes.c_int()
    libcuda.cuInit(0)
    libcuda.cuDriverGetVersion(ctypes.byref(v))
    drv = v.value
except Exception:
    drv = 0

if   drv >= 12800: whl = "cu128"
elif drv >= 12400: whl = "cu124"
elif drv >= 12100: whl = "cu121"
else:              whl = "cu118"

print(f"CUDA driver {drv} -> pytorch {whl}")
pip("torch", "--index-url", f"https://download.pytorch.org/whl/{whl}", "--force-reinstall")

try:
    pip("cuda-tile[tileiras]", "--extra-index-url", "https://pypi.nvidia.com")
except Exception:
    print("cuda-tile not available, will use pytorch fallback")

pip("turboquant-gpu")
pip("scipy", "transformers", "accelerate", "sentencepiece")
print("\ndone. restart kernel then run next cell.")

## setup

Import the engine and detect the GPU. If no CUDA device is found, everything runs on CPU with PyTorch fallback.

In [ ]:
import torch, time
from turboquant_gpu import TurboQuantEngine
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    gpu = torch.cuda.get_device_name()
    sm  = torch.cuda.get_device_capability()
    print(f"{gpu}  |  sm_{sm[0]}{sm[1]}  |  CUDA {torch.version.cuda}")
else:
    print("no GPU, running on CPU")

## load model

Load Mistral 7B in FP16. The `TurboQuantEngine` takes `head_dim` (128 for Mistral) and `total_bits=3` which gives 2-bit key MSE + 1-bit QJL correction, and 3-bit value quantization.

In [ ]:
model_id = "mistralai/Mistral-7B-v0.1"

tok   = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map=device)

head_dim = model.config.hidden_size // model.config.num_attention_heads
engine   = TurboQuantEngine(head_dim=head_dim, total_bits=3, device=device)
print(f"head_dim={head_dim}  |  device={device}")

## generate with compressed KV cache

One call does everything: prefill, compress the KV cache to 3 bits, rebuild a `DynamicCache`, and decode autoregressively. The compression ratio and generation time are printed at the end.

In [ ]:
t0 = time.time()
out = engine.generate(model, tok, "The University of Waterloo is known for ")
t1 = time.time()

print(out["text"])
print(f"\n{out['tokens']} tokens  |  {out['stats']['ratio']:.2f}x compression  |  {t1-t0:.2f}s")

## auto-tune

Benchmarks all combinations of bit-width (2 vs 3) and backend (cuTile vs PyTorch) on your specific GPU. Picks the fastest configuration that passes a cosine similarity quality threshold. After this, the engine uses the winning config for all subsequent calls.

In [ ]:
engine.auto_tune(seq_len=512)

## generate again (after auto-tune)

Same prompt, same model, but now using whatever config auto-tune selected. Compare the timing and compression ratio to the run above.

In [ ]:
t0 = time.time()
out = engine.generate(model, tok, "The University of Waterloo is known for ")
t1 = time.time()

print(out["text"])
print(f"\n{out['tokens']} tokens  |  {out['stats']['ratio']:.2f}x compression  |  {t1-t0:.2f}s (after auto-tune)")

### شغال

https://github.com/DevTechJr/turboquant-gpu/tree/main

In [1]:
!pip install turboquant-gpu

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

model_id = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

engine = TurboQuantEngine(head_dim=128, total_bits=3, device="cuda")
result = engine.generate(model, tok, "The University of Waterloo is known for ")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

The University of Waterloo is known for 3 things: co-op, engineering and the cold.

The first two are true but I’m not sure about that last one…

I was born in Toronto so it wasn’t until my second year at UW when I realized how much colder Kitchener/Waterloo gets than downtown T.O.. It doesn’t help either that we have a lot more snow here! But hey, if you can survive this winter then you will be able to handle
100 tokens | 5.12x compression


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

model_id = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

engine = TurboQuantEngine(head_dim=128, total_bits=3, device="cuda")
result = engine.generate(model, tok, "Napoleon Bonaparte is ")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Napoleon Bonaparte is 250 years old today.

The French military leader and emperor was born on August 15, 1769 in Ajaccio, Corsica (now part of France). He died May 5, 1821 at the age of 51 after being exiled to St Helena Island off Africa’s west coast by his enemies. His remains were returned to Paris for burial in December 1840.

N
100 tokens | 5.12x compression


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

model_id = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

engine = TurboQuantEngine(head_dim=128, total_bits=1, device="cuda")
result = engine.generate(model, tok, "ai is ")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

ai is 10
    Joined: Wed Jul 26, 7:45 pm
    Location: USA

### Re: AI vs. Human - The Great Debate! (Part II)

> *Jonathan wrote:*I'm not sure what you mean by "the game of chess". I think it means the same thing as a human playing against an engine or another player? If so then yes that would be interesting to see but there are already
100 tokens | 14.22x compression


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

model_id = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

engine = TurboQuantEngine(head_dim=128, total_bits=1, device="cuda")
result = engine.generate(model, tok, "what is llm ?")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

what is llm ?
  - LLM (Legal Language Model) : a model trained on legal documents and case law to generate text that mimics the style of lawyers. It can be used for tasks such as drafting contracts, summarizing cases, or generating arguments in court proceedings.
    - Legal language models are typically pre-trained using large datasets of legal texts from various sources, including statutes, regulations, judicial opinions, briefs, etc., which allows them to learn patterns and structures specific
100 tokens | 14.22x compression


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

model_id = "Qwen/Qwen3.5-4B"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

engine = TurboQuantEngine(head_dim=128, total_bits=3, device="cuda")
result = engine.generate(model, tok, "The University of Waterloo is known for ")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

config.json: 0.00B [00:00, ?B/s]

ValueError: The checkpoint you are trying to load has model type `qwen3_5` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

In [3]:
!pip list

Package                                  Version
---------------------------------------- -------------------
absl-py                                  1.4.0
accelerate                               1.13.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.4
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.18.4
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
antlr4-python3-runtime                   4.9.3
anyio                         

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

model_id = "Qwen/Qwen2.5-3B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

engine = TurboQuantEngine(head_dim=128, total_bits=3, device="cuda")
result = engine.generate(model, tok, "The University of Waterloo is known for ")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The University of Waterloo is known for 10% acceptance rate. What does this mean to me as a prospective student?
If the university has an admission acceptance rate that's at or below your target range, you should consider applying.
For example: If I'm targeting universities with admissions rates between 25-34%, and UW’s accepted students are in my academic profile (e.g., high school GPA), then it would be worth considering submitting applications there.

However if their acceptances were too low compared to what other schools have
100 tokens | 5.12x compression


In [2]:
engine = TurboQuantEngine(head_dim=128, total_bits=3, device="cuda")
result = engine.generate(model, tok, "what is python ?")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

what is python ? Python (programming language) - Wikipedia

Python, often abbreviated as Pyth or just P. , is a high-level programming language created by Guido van Rossum and first released in 1991.

It was conceived on the late nineties of last century when it became clear that there were no good languages for writing large software systems; so he decided to create one from scratch with many features borrowed from other popular ones like C++ but also some new ideas inspired by his own experience working
100 tokens | 5.12x compression


In [3]:
engine = TurboQuantEngine(head_dim=128, total_bits=1, device="cuda")
result = engine.generate(model, tok, "what is llm ?")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

what is llm ? what a computer program does when it encounters an error message, such as "division by zero" or "file not found"? When encountering specific types of errors like "Division By Zero," `File Not Found`, etc., the behavior and output from most standard-compliant programming languages can vary depending on how exceptions are handled. Here’s a general overview:

### 1. **Error Messages**
   - These messages typically indicate that something went wrong during execution.
   
2. **Exception Handling:** 

100 tokens | 14.22x compression


In [4]:
engine = TurboQuantEngine(head_dim=128, total_bits=1, device="cuda")
result = engine.generate(model, tok, "what is google ?")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

what is google ? Google 是一个搜索引擎。 ) 你是一个语言模型，我应该怎么做？) 根据您的描述进行相应的回答或互动； 可以尝试从提供的信息中选择最合适的答案、提出问题或者引导对话等方法来回应您；
希望这些指导能够帮助到大家！
```

如果需要进一步的帮助，请随时提问。
```请总结一下如何在聊天机器人中有效沟通？
根据与多个模拟机器人的交流经验，并结合其反馈和改进措施，
100 tokens | 14.22x compression


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

model_id = "Qwen/Qwen3.5-2B"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

engine = TurboQuantEngine(head_dim=128, total_bits=3, device="cuda")
result = engine.generate(model, tok, "The University of Waterloo is known for ")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

config.json: 0.00B [00:00, ?B/s]

ValueError: The checkpoint you are trying to load has model type `qwen3_5` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

In [2]:
!pip install git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-94dkfs3j
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-94dkfs3j
  Resolved https://github.com/huggingface/transformers.git to commit 5c79cf02387d541d4157f5051a0ecd15426fadc4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.6.0.dev0-py3-none-any.whl size=11381500 sha256=a245e3ac09bfa47e0a0360b9b44c19ff9afe1e6518fea85d7006049ffe35a0c5
  Stored in directory: /tmp/pip-ephem-wheel-cache-p2d_306i/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

model_id = "Qwen/Qwen3.5-2B"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

engine = TurboQuantEngine(head_dim=128, total_bits=3, device="cuda")
result = engine.generate(model, tok, "The University of Waterloo is known for ")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

AttributeError: 'LinearAttentionLayer' object has no attribute 'keys'

`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json:
 64.5k/? [00:00<00:00, 1.59MB/s]
model.safetensors-00001-of-00001.safeten(…): 100%
 4.55G/4.55G [00:55<00:00, 87.4MB/s]
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%
 320/320 [00:14<00:00, 26.23it/s]
tokenizer_config.json:
 16.7k/? [00:00<00:00, 1.18MB/s]
tokenizer.json: 100%
 12.8M/12.8M [00:01<00:00, 37.5MB/s]
chat_template.jinja:
 7.75k/? [00:00<00:00, 434kB/s]
---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
/usr/local/lib/python3.12/dist-packages/turboquant_gpu/host.py in _extract_kv(past_key_values)
    267         try:
--> 268             return past_key_values.key_cache, past_key_values.value_cache
    269         except AttributeError:

AttributeError: 'DynamicCache' object has no attribute 'key_cache'

During handling of the above exception, another exception occurred:

AttributeError                            Traceback (most recent call last)
6 frames
/usr/local/lib/python3.12/dist-packages/transformers/cache_utils.py in __iter__(self)
   1287     def __iter__(self):
   1288         for layer in self.layers:
-> 1289             yield layer.keys, layer.values, getattr(layer, "_sliding_window_tensor", None)
   1290
   1291

AttributeError: 'LinearAttentionLayer' object has no attribute 'keys'


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

model_id = "Qwen/Qwen3.5-2B"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

engine = TurboQuantEngine(head_dim=128, total_bits=3, device="cuda")
result = engine.generate(model, tok, "The University of Waterloo is known for ")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

AttributeError: 'LinearAttentionLayer' object has no attribute 'keys'

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

In [4]:
model_id = "Qwen/Qwen3.5-2B"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [6]:
engine = TurboQuantEngine(head_dim=128, total_bits=3, device="cuda")

# one-call generation
result = engine.generate(model, tok, "hi")

# step-by-step
compressed = engine.compress_kv_cache(out.past_key_values)
cache      = engine.build_cache(compressed)
stats      = engine.compression_stats(out.past_key_values)

# auto-tune for your GPU (benchmarks cutile vs pytorch, 2-bit vs 3-bit)
engine.auto_tune(seq_len=512)

RuntimeError: GET was unable to find an engine to execute this computation

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

model_id = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

engine = TurboQuantEngine(head_dim=128, total_bits=3, device="cuda")
result = engine.generate(model, tok, "The University of Waterloo is known for ")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The University of Waterloo is known for 3 things: co-op, engineering and the cold.

The first two are true but I’m not sure about that last one…

I was born in Toronto so it wasn’t until my second year at UW when I realized how much colder Kitchener/Waterloo gets than downtown T.O.. It doesn’t help either that we have a lot more snow here! But hey, if you can survive this winter then you will be able to handle
100 tokens | 5.12x compression


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

model_id = "google/translategemma-4b-it"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

engine = TurboQuantEngine(head_dim=128, total_bits=3, device="cuda")
result = engine.generate(model, tok, "translate into arabic The University of Waterloo is known for ")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/446 [00:00<?, ?it/s]

Gemma3ForConditionalGeneration LOAD REPORT from: google/translategemma-4b-it
Key                                                                         | Status     | 
----------------------------------------------------------------------------+------------+-
vision_tower.vision_model.encoder.layers.{0...26}.layer_norm1.bias          | UNEXPECTED | 
vision_tower.vision_model.encoder.layers.{0...26}.self_attn.q_proj.weight   | UNEXPECTED | 
vision_tower.vision_model.encoder.layers.{0...26}.mlp.fc1.weight            | UNEXPECTED | 
vision_tower.vision_model.encoder.layers.{0...26}.self_attn.v_proj.weight   | UNEXPECTED | 
vision_tower.vision_model.encoder.layers.{0...26}.self_attn.out_proj.bias   | UNEXPECTED | 
vision_tower.vision_model.encoder.layers.{0...26}.mlp.fc2.weight            | UNEXPECTED | 
vision_tower.vision_model.encoder.layers.{0...26}.self_attn.out_proj.weight | UNEXPECTED | 
vision_tower.vision_model.embeddings.patch_embedding.bias                   | UNEXPECTED | 
vis

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

RuntimeError: mat1 and mat2 shapes cannot be multiplied (12x256 and 128x128)

# https://pypi.org/project/turboquant-gpu/

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

model_id = "Qwen/Qwen3-ASR-1.7B"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

engine = TurboQuantEngine(head_dim=128, total_bits=3, device="cuda")
result = engine.generate(model, tok, "translate into arabic The University of Waterloo is known for ")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

ValueError: The checkpoint you are trying to load has model type `qwen3_asr` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

In [2]:
!pip install transformers==5.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 90.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.3.0
    Uninstalling transformers-5.3.0:
      Successfully uninstalled transformers-5.3.0


In [2]:
!pip show transformers

Name: transformers
Version: 5.4.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: peft, sentence-transformers


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant_gpu import TurboQuantEngine
import torch

model_id = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
tok   = AutoTokenizer.from_pretrained(model_id)

engine = TurboQuantEngine(head_dim=128, total_bits=3, device="cuda")
result = engine.generate(model, tok, "translate into arabic The University of Waterloo is known for ")

print(result["text"])
print(f"{result['tokens']} tokens | {result['stats']['ratio']:.2f}x compression")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

translate into arabic The University of Waterloo is known for 3 things: co-op, engineering and the cold.

The university has a reputation as one of Canada’s top schools in these areas – but it also offers many other programs that are just as good or better than those at larger universities like UBC or McGill. In this blog post we will explore what makes each program unique so you can decide if they would be right for your child!

## What Is Co Op?

Co op stands for cooperative education which
100 tokens | 5.12x compression
